In [ ]:
!pip install rouge-score bert-score evaluate

In [ ]:
!pip uninstall -y torchao
!pip install -U torchao

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import json
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch
from collections import defaultdict
import os
import gc
import evaluate
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
folder_path = '/content/drive/MyDrive/facetsum/finetunning_data/test.json'

with open(folder_path, "r", encoding="utf-8") as f:
    data = json.load(f)

In [ ]:
sample = data[0:30]

In [ ]:
!hf auth login

In [ ]:
def text_to_json(text):
  text = re.sub(r"```json\s*", "", text)
  text = re.sub(r"\s*```", "", text)
  text = text.strip()

  try:
      return json.loads(text)
  except json.JSONDecodeError:
      pass

  pattern = (
    r"Purpose:\s*-?\s*(.*?)"
    r"Method:\s*-?\s*(.*?)"
    r"Findings:\s*-?\s*(.*?)"
    r"Value:\s*-?\s*(.*)"
)

  match = re.search(pattern, text, flags=re.DOTALL | re.IGNORECASE)
  if not match:
      return None

  data = {"Purpose": f"{match.group(1).strip()}",
          "Method": f"{match.group(2).strip()}",
          "Findings": f"{match.group(3).strip()}",
          "Value": f"{match.group(4).strip()}",}

  return data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")

In [ ]:
def generate_output(input, model):
  message = [
    {
        "role": "system",
        "content": input['system']
    },
    {
        "role": "user",
        "content":  input['instruction']
    }
]
  inputs = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    ).to(model.device)


  with torch.inference_mode():
    outputs = model.generate(**inputs, max_new_tokens=1024)

  answer = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
  del inputs
  del outputs

  torch.cuda.empty_cache()
  gc.collect()

  return text_to_json(answer)

In [ ]:
def save_evaluation_set_answers(eval_set, model, valid_data_path):

  for rec in eval_set :

    answers = {}
    try :

      model_answer = generate_output(rec, model)
      answers['model_answer'] = model_answer

      refrence_answer = text_to_json(rec["output"])
      answers['refrence_answer'] = refrence_answer

      with open(valid_data_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(answers, ensure_ascii=False) + "\n")

    except Exception as e:
      with open(oom_data_path, "a", encoding="utf-8") as f:
        f.write(json.dumps(answers, ensure_ascii=False) + "\n")


In [ ]:
rouge = evaluate.load("rouge")
sections = ["Purpose", "Method", "Findings", "Value"]

In [ ]:
def flatten(value):
  if isinstance(value, str):
    return value

  if isinstance(value, dict):
    return " ".join(flatten(v) for v in value.values())

  if isinstance(value, list):
    return " ".join(flatten(v) for v in value)

  return str(value)

In [ ]:
def compute_rouge_per_section(valid_data_path):
  scores = {s: defaultdict(list) for s in sections}

  with open(valid_data_path, "r", encoding="utf-8") as f:
    for line in f:
      try:
        sample = json.loads(line)

        ref = sample["refrence_answer"]
        pred = sample["model_answer"]

        if isinstance(pred, list):
          pred = pred[0]

        for sec in sections:

          pred_text = flatten(pred.get(sec, ""))
          ref_text = flatten(ref.get(sec, ""))

          result = rouge.compute(
              predictions=[pred_text],
              references=[ref_text],
              use_stemmer=True
          )

          scores[sec]["rougeL"].append(result["rougeL"])

      except Exception as e:
        print(f"error: {e}")
        continue

  final_scores = {}
  for sec in sections:
    final_scores[sec] = {
        "rougeL_mean": sum(scores[sec]["rougeL"]) / len(scores[sec]["rougeL"])
    }

  return final_scores

In [ ]:
def compute_bertscore_per_section(valid_data_path):
  scores = {s: defaultdict(list) for s in sections}

  with open(valid_data_path, "r", encoding="utf-8") as f:
    for line in f:
      try:
        sample = json.loads(line)

        ref = sample["refrence_answer"]
        pred = sample["model_answer"]

        if isinstance(pred, list):
          pred = pred[0]

        for sec in sections:

          pred_text = pred.get(sec, "")
          ref_text = ref.get(sec, "")

          P, R, F1 = bert_score(
              [pred_text],
              [ref_text],
              lang="en",
              model_type="distilbert-base-uncased"
          )

          scores[sec]["bertscore_f1"].append(F1.item())

      except Exception as e:
        print(f"error: {e}")
        continue

  final_scores = {}

  for sec in sections:
    final_scores[sec] = {
        "bertscore_f1_mean":
            sum(scores[sec]["bertscore_f1"]) /
            len(scores[sec]["bertscore_f1"])
    }

  return final_scores

## Evaluate model before finetunning by RougeL and BertScore

In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    device_map="auto",
)


In [ ]:
base_dir = "/content/drive/MyDrive/facetsum/dataset"
os.makedirs(base_dir, exist_ok=True)

valid_data_path = os.path.join(base_dir, "test_evaluation.jsonl")
oom_data_path = os.path.join(base_dir, "test_oom_evaluation.jsonl")

In [ ]:
save_evaluation_set_answers(sample, model, valid_data_path)

In [1]:
final_scores = compute_rouge_per_section(valid_data_path)
final_scores

{'Purpose': {'rougeL_mean': np.float64(0.2780366654512784)},
 'Method': {'rougeL_mean': np.float64(0.23468469420314478)},
 'Findings': {'rougeL_mean': np.float64(0.22387679541601616)},
 'Value': {'rougeL_mean': np.float64(0.18182365922665386)}}


In [ ]:
final_scores = compute_bertscore_per_section(valid_data_path)

In [ ]:
final_scores

{'Purpose': {'bertscore_f1_mean': 0.8207270879494516},
 'Method': {'bertscore_f1_mean': 0.8055421176709627},
 'Findings': {'bertscore_f1_mean': 0.8024346828460693},
 'Value': {'bertscore_f1_mean': 0.8000461283852073}}

## Evaluate model after finetunning by RougeL and BertScore

In [ ]:
finetuned_model_id = "/content/drive/MyDrive/facetsum/test_model"
model.load_adapter(finetuned_model_id)

In [ ]:
base_dir = "/content/drive/MyDrive/facetsum/finetunning_data"
os.makedirs(base_dir, exist_ok=True)

valid_data_path = os.path.join(base_dir, "test_evaluation.jsonl")
oom_data_path = os.path.join(base_dir, "test_oom_evaluation.jsonl")

In [ ]:
save_evaluation_set_answers(sample, model, valid_data_path)

In [ ]:
final_scores = compute_rouge_per_section(valid_data_path)
final_scores

{'Purpose': {'rougeL_mean': np.float64(0.3642276024297903)},
 'Method': {'rougeL_mean': np.float64(0.23355662444707417)},
 'Findings': {'rougeL_mean': np.float64(0.2258458389074259)},
 'Value': {'rougeL_mean': np.float64(0.24597588154030323)}}

In [ ]:
final_scores = compute_bertscore_per_section(valid_data_path)

In [ ]:
final_scores

{'Purpose': {'bertscore_f1_mean': 0.8568289705685207},
 'Method': {'bertscore_f1_mean': 0.8128222085180736},
 'Findings': {'bertscore_f1_mean': 0.8048021850131807},
 'Value': {'bertscore_f1_mean': 0.8129314184188843}}